### Graph Neural Networks & Blood Brain Barrier Permeability

In [1]:
import pandas as pd
import torch
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

import sys, os
sys.path.append(os.path.abspath("../"))
from src.featurization import df_to_pyg_lists
from src.gnn_model import BBBP_GCN

In [2]:
# load dataset

df = pd.read_csv(r"../data/processed/neuro_natural_dataset.csv")

In [3]:
train_graphs, test_graphs = df_to_pyg_lists(df)

[22:35:24] WARNING: not removing hydrogen atom without neighbors
[22:35:24] WARNING: not removing hydrogen atom without neighbors
[22:35:24] WARNING: not removing hydrogen atom without neighbors
[22:35:24] WARNING: not removing hydrogen atom without neighbors
[22:35:24] WARNING: not removing hydrogen atom without neighbors
[22:35:24] WARNING: not removing hydrogen atom without neighbors
[22:35:24] WARNING: not removing hydrogen atom without neighbors
[22:35:24] WARNING: not removing hydrogen atom without neighbors
[22:35:25] WARNING: not removing hydrogen atom without neighbors
[22:35:25] WARNING: not removing hydrogen atom without neighbors
[22:35:25] WARNING: not removing hydrogen atom without neighbors
[22:35:25] WARNING: not removing hydrogen atom without neighbors
[22:35:25] WARNING: not removing hydrogen atom without neighbors
[22:35:25] WARNING: not removing hydrogen atom without neighbors
[22:35:25] WARNING: not removing hydrogen atom without neighbors
[22:35:25] WARNING: not r

In [4]:
len(train_graphs), len(test_graphs)

(1304, 327)

In [6]:
# data loaders
batch_size = 64
train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

In [7]:
# model, optimizer, loss (with class-imbalance weighting)

# pick device: MPS on Apple Silicon if available, else CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device:", device)
else:
    device = torch.device("cpu")
    print("Using device:", device)

in_channels = train_graphs[0].x.size(1)
model = BBBP_GCN(in_channels=in_channels, hidden_dim=64).to(device)

# class weights to handle imbalance (0 = non-perm, 1 = perm)
class_counts = df["permeable"].value_counts()
total = class_counts.sum()
w0 = total / (2 * class_counts[0.0])
w1 = total / (2 * class_counts[1.0])
class_weights = torch.tensor([w0, w1], dtype=torch.float).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

Using device: mps


In [8]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = (probs >= 0.5).long()

        all_probs.append(probs.cpu())
        all_preds.append(preds.cpu())
        all_labels.append(batch.y.cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    return {
        "auc": roc_auc_score(all_labels, all_probs),
        "f1": f1_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds),
        "recall": recall_score(all_labels, all_preds),
        "accuracy": accuracy_score(all_labels, all_preds),
    }

num_epochs = 40
for epoch in range(1, num_epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    if epoch % 5 == 0 or epoch == 1:
        metrics = eval_model(model, test_loader, device)
        print(
            f"Epoch {epoch:02d} | loss={train_loss:.4f} | "
            f"AUC={metrics['auc']:.3f} | F1={metrics['f1']:.3f} | "
            f"P={metrics['precision']:.3f} | R={metrics['recall']:.3f}"
        )

Epoch 01 | loss=0.6837 | AUC=0.757 | F1=0.910 | P=0.851 | R=0.978
Epoch 05 | loss=0.5673 | AUC=0.787 | F1=0.797 | P=0.922 | R=0.703
Epoch 10 | loss=0.5240 | AUC=0.797 | F1=0.846 | P=0.917 | R=0.784
Epoch 15 | loss=0.5015 | AUC=0.811 | F1=0.834 | P=0.931 | R=0.755
Epoch 20 | loss=0.4998 | AUC=0.819 | F1=0.791 | P=0.930 | R=0.688
Epoch 25 | loss=0.4723 | AUC=0.824 | F1=0.893 | P=0.935 | R=0.855
Epoch 30 | loss=0.4522 | AUC=0.830 | F1=0.887 | P=0.934 | R=0.844
Epoch 35 | loss=0.4554 | AUC=0.838 | F1=0.881 | P=0.930 | R=0.836
Epoch 40 | loss=0.4390 | AUC=0.846 | F1=0.857 | P=0.934 | R=0.792
